## Regridding with `Regrid`

The `Regrid` resampler converts data from a **HEALPix** or **reduced Gaussian** source grid to a regular latitude/longitude grid in *data space* before plotting, using earthkit-geo under the hood. Because the conversion happens before rendering, the result is independent of the target map CRS and can be further processed by any pixel-space resampler.

Use `Regrid` when:
- your source data is on a **HEALPix** or **reduced Gaussian (octahedral)** grid, and
- you want to plot it with methods that expect a regular lat/lon grid (e.g. `contourf`, `pcolormesh`).

`Regrid` supports two interpolation methods:
- `'linear'` (default) — bilinear interpolation in data space, producing smooth output.
- `'nearest-neighbour'` — nearest cell lookup, preserving exact grid-cell values.

<div class="alert alert-block alert-info">
<strong>NOTE:</strong> <code>Regrid</code> requires the <code>earthkit-geo</code> package (with MIR support). For regular lat/lon data, use <a href="resampling-bilinear.ipynb">Bilinear</a> or <a href="resampling-nearest-neighbour.ipynb">NearestNeighbour</a> instead — <code>Regrid</code> will raise an error if given a regular grid.
</div>

### Example: HEALPix 2 m temperature

We load a HEALPix GRIB file at H128 resolution (nested ordering) containing 2 m temperature, then regrid it to a regular 0.5° lat/lon grid before plotting.

In [ ]:
import earthkit.data as ekd

import earthkit.plots as ekp
from earthkit.plots.resample import Regrid

data = ekd.from_source("sample", "healpix-h128-nested-2t.grib")

chart = ekp.Map(domain="Europe")

# Regrid to 0.5° lat/lon using linear interpolation
chart.contourf(
    data,
    resample=Regrid(resolution=0.5),
    style=ekp.styles.Style(
        levels=range(240, 310, 5),
        colors="Spectral_r",
    ),
)

chart.coastlines()
chart.gridlines()
chart.legend()

chart.show()

### Choosing the output resolution

The `resolution` parameter controls the spacing (in degrees) of the regular lat/lon output grid. A finer resolution produces more detail but takes longer to compute. The default is `0.2°`.

In [ ]:
style = ekp.styles.Style(
    levels=range(240, 310, 5),
    colors="Spectral_r",
)

figure = ekp.Figure(rows=1, columns=2, domain="Europe")

ax = figure.add_map()
ax.contourf(data, resample=Regrid(resolution=2.0), style=style)
ax.title("resolution=2.0°")

ax = figure.add_map()
ax.contourf(data, resample=Regrid(resolution=0.25), style=style)
ax.title("resolution=0.25°")

figure.coastlines()
figure.legend(location="right")

figure.show()

### Nearest-neighbour method

Pass `method='nearest-neighbour'` (or the alias `'nearest'`) to use a nearest-cell lookup instead of linear interpolation. This preserves the exact grid-cell values and shows the native HEALPix pixel boundaries.

In [ ]:
figure = ekp.Figure(rows=1, columns=2, domain="Europe")

ax = figure.add_map()
ax.contourf(data, resample=Regrid(resolution=0.5, method="linear"), style=style)
ax.title("method='linear'")

ax = figure.add_map()
ax.contourf(data, resample=Regrid(resolution=0.5, method="nearest"), style=style)
ax.title("method='nearest-neighbour'")

figure.coastlines()
figure.legend(location="right")

figure.show()

### Supplying the grid spec manually

If your data does not carry the grid metadata that earthkit-plots needs (for example, after converting to xarray and stripping the `_earthkit` attribute), you can supply the source grid specification explicitly via the `in_grid` parameter.

In [ ]:
# Convert to xarray and strip the earthkit metadata
ring_data = ekd.from_source("sample", "healpix-h128-ring-2t.grib")
ds = ring_data.to_xarray()
ds.t.attrs.pop("_earthkit", None)

chart = ekp.Map(domain="Europe")

# Tell Regrid what the source grid is
chart.contourf(
    ds,
    resample=Regrid(resolution=0.5, in_grid={"grid": "H128", "order": "ring"}),
    style=style,
)

chart.coastlines()
chart.gridlines()
chart.legend()

chart.show()

### Combining Regrid with pixel-space resamplers

Because `Regrid` converts data to a regular lat/lon grid first, you can chain it with `Bilinear` or `NearestNeighbour` for additional pixel-space smoothing. Use the `Chain` resampler for this — see the [Regrid notebook](resampling-regrid.ipynb) for more detail.

In [ ]:
from earthkit.plots.resample import Bilinear, Chain

chart = ekp.Map(domain="Europe")

# Regrid to 1° lat/lon, then smooth with bilinear pixel resampling
chart.contourf(
    data,
    resample=Chain(Regrid(resolution=1.0), Bilinear(500)),
    style=style,
)

chart.coastlines()
chart.gridlines()
chart.legend()

chart.show()